# Smart Gallery backend

Serves the fine-tuned model to the app. It stores no photos and no index: the phone keeps the vectors.

**To go live:** add secrets `NGROK_AUTH_TOKEN` and `SMART_GALLERY_API_KEY` (Add-ons → Secrets), and optionally
`NGROK_DOMAIN`. Without them the notebook runs its self-test and stops.

In [ ]:
!pip install -q open_clip_torch==3.3.0 transformers sentencepiece protobuf fastapi uvicorn python-multipart httpx pyngrok

import io, json, os, subprocess, sys, threading, time
from pathlib import Path

import numpy as np
import torch

print('GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

# How long to stay up. Every hour here spends GPU quota, so idle sessions stop themselves.
MAX_HOURS = 11
IDLE_STOP_MINUTES = 60
RUN_SELF_TEST = True
SELF_TEST_IMAGES = 1000

In [ ]:
%%writefile server.py
"""Smart Gallery backend: a stateless image and text embedding service.

The server keeps no photos and no index. The app sends small thumbnails, stores the
vectors it gets back, and ranks them on the phone, so a Kaggle restart loses nothing.

    GET  /health        model details, no key needed
    POST /embed/images  multipart "files", up to MAX_FILES images
    POST /embed/text    {"queries": ["a dog in the snow"]}

Vectors travel as base64 int8 (QUANT_SCALE * value, clipped to +-127): 1 KB per photo
instead of 4 KB, for a loss in ranking measured by the notebook's self-test.
"""
import base64
import hashlib
import hmac
import io
import threading
import time
from collections import OrderedDict
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import torch
from fastapi import Depends, FastAPI, File, Header, HTTPException, UploadFile
from fastapi.concurrency import run_in_threadpool
from PIL import Image, ImageFile, ImageOps
from pydantic import BaseModel, Field

from smart_gallery import load_checkpoint

ImageFile.LOAD_TRUNCATED_IMAGES = True

MAX_FILES = 32
MAX_FILE_BYTES = 8 * 1024 * 1024
MAX_QUERIES = 16
MAX_QUERY_CHARS = 300
QUANT_SCALE = 512.0
TEXT_CACHE_SIZE = 4096


def quantize(vectors):
    return np.clip(np.rint(vectors * QUANT_SCALE), -127, 127).astype(np.int8)


def encode_vector(vector):
    return base64.b64encode(vector.tobytes()).decode("ascii")


def model_fingerprint(path):
    """Identifies the weights. The app rebuilds its index when this changes."""
    path = Path(path)
    digest = hashlib.sha256(str(path.stat().st_size).encode())
    with open(path, "rb") as f:
        digest.update(f.read(1024 * 1024))
    return digest.hexdigest()[:12]


class Embedder:
    """The image tower runs on the first GPU and the text tower on the second, so
    searches stay fast while a phone is indexing."""

    def __init__(self, checkpoint, decode_workers=4):
        count = torch.cuda.device_count()
        self.image_device = torch.device("cuda:0" if count else "cpu")
        self.text_device = torch.device(f"cuda:{count - 1}" if count else "cpu")
        self.model, self.preprocess, self.tokenizer, self.source = load_checkpoint(checkpoint, self.image_device)
        self.model.text.to(self.text_device)
        self.version = model_fingerprint(checkpoint)
        self.image_lock = threading.Lock()
        self.text_lock = threading.Lock() if count > 1 else self.image_lock
        self.decoders = ThreadPoolExecutor(decode_workers)
        self.text_cache = OrderedDict()
        self.dim = len(self.embed_texts(["a photo"])[0])

    def _autocast(self, device):
        return torch.autocast("cuda", enabled=device.type == "cuda")

    def _decode(self, data):
        try:
            image = ImageOps.exif_transpose(Image.open(io.BytesIO(data))).convert("RGB")
            return self.preprocess(image)
        except Exception:
            return None

    def embed_image_floats(self, images):
        """Returns one vector per input, or None for an image that could not be read."""
        tensors = list(self.decoders.map(self._decode, images))
        usable = [i for i, tensor in enumerate(tensors) if tensor is not None]
        results = [None] * len(images)
        if not usable:
            return results
        batch = torch.stack([tensors[i] for i in usable])
        with self.image_lock, torch.inference_mode(), self._autocast(self.image_device):
            vectors = self.model.encode_image(batch.to(self.image_device), normalize=True)
        vectors = vectors.float().cpu().numpy()
        for index, vector in zip(usable, vectors):
            results[index] = vector
        return results

    def embed_texts(self, queries):
        with self.text_lock:
            found = {}
            missing = []
            for query in dict.fromkeys(queries):
                if query in self.text_cache:
                    self.text_cache.move_to_end(query)
                    found[query] = self.text_cache[query]
                else:
                    missing.append(query)
            if missing:
                tokens = self.tokenizer(missing).to(self.text_device)
                with torch.inference_mode(), self._autocast(self.text_device):
                    vectors = self.model.encode_text(tokens, normalize=True)
                for query, vector in zip(missing, vectors.float().cpu().numpy()):
                    found[query] = vector
                    self.text_cache[query] = vector
                while len(self.text_cache) > TEXT_CACHE_SIZE:
                    self.text_cache.popitem(last=False)
        return [found[query] for query in queries]


class TextRequest(BaseModel):
    queries: list[str] = Field(min_length=1, max_length=MAX_QUERIES)


def create_app(embedder, api_key):
    if not api_key:
        raise ValueError("An API key is required: the server is reachable from the internet.")

    app = FastAPI(title="Smart Gallery", docs_url=None, redoc_url=None)
    app.state.last_request = time.time()
    app.state.embedded_images = 0

    def require_key(x_api_key: str = Header(default="")):
        if not hmac.compare_digest(x_api_key.encode(), api_key.encode()):
            raise HTTPException(401, "Invalid API key")
        app.state.last_request = time.time()

    @app.get("/health")
    def health():
        return {
            "status": "ok",
            "model": embedder.source[0],
            "model_version": embedder.version,
            "dim": embedder.dim,
            "quant_scale": QUANT_SCALE,
            "max_files": MAX_FILES,
            "gpus": torch.cuda.device_count(),
        }

    @app.post("/embed/images", dependencies=[Depends(require_key)])
    async def embed_images(files: list[UploadFile] = File()):
        if len(files) > MAX_FILES:
            raise HTTPException(413, f"Send at most {MAX_FILES} images at once")
        images = []
        for file in files:
            data = await file.read(MAX_FILE_BYTES + 1)
            if len(data) > MAX_FILE_BYTES:
                raise HTTPException(413, f"{file.filename} is too large; send a thumbnail")
            images.append(data)
        vectors = await run_in_threadpool(embedder.embed_image_floats, images)
        app.state.embedded_images += sum(vector is not None for vector in vectors)
        return {
            "model_version": embedder.version,
            "vectors": [None if v is None else encode_vector(quantize(v)) for v in vectors],
        }

    @app.post("/embed/text", dependencies=[Depends(require_key)])
    async def embed_text(request: TextRequest):
        queries = [query.strip().lower()[:MAX_QUERY_CHARS] for query in request.queries]
        if not all(queries):
            raise HTTPException(422, "Queries must not be empty")
        vectors = await run_in_threadpool(embedder.embed_texts, queries)
        return {
            "model_version": embedder.version,
            "vectors": [encode_vector(quantize(vector)) for vector in vectors],
        }

    return app

In [ ]:
checkpoints = list(Path('/kaggle/input').rglob('smart_gallery_model.pt'))
assert checkpoints, 'Attach the output of the smart-gallery-search notebook as an input.'
checkpoint = checkpoints[0]
sys.path.insert(0, str(checkpoint.parent))
sys.path.insert(0, '.')
print('Model:', checkpoint, f'({checkpoint.stat().st_size / 1e9:.2f} GB)')

from server import Embedder, create_app, QUANT_SCALE, quantize

embedder = Embedder(checkpoint)
print(embedder.source, 'dim', embedder.dim, 'version', embedder.version)

## Self-test

Runs the same path the phone uses: 384px JPEGs through the real HTTP API, compressed vectors, ranking by dot product.

In [ ]:
import base64
import random

from fastapi.testclient import TestClient
from PIL import Image

def find_coco():
    for root, _, files in os.walk('/kaggle/input'):
        if 'captions_val2017.json' in files:
            annotations = os.path.join(root, 'captions_val2017.json')
            break
    else:
        return None
    images = next(d for d, _, f in os.walk('/kaggle/input') if os.path.basename(d) == 'val2017')
    return annotations, images

def thumbnail(path):
    # Matches the app: a square 384px JPEG at quality 80.
    image = Image.open(path).convert('RGB').resize((384, 384), Image.BICUBIC)
    buffer = io.BytesIO()
    image.save(buffer, 'JPEG', quality=80)
    return buffer.getvalue()

def decode(vector):
    return np.frombuffer(base64.b64decode(vector), dtype=np.int8)

def self_test():
    key = 'self-test-key'
    client = TestClient(create_app(embedder, key))
    headers = {'X-API-Key': key}

    health = client.get('/health').json()
    assert health['status'] == 'ok' and health['dim'] == embedder.dim, health
    assert client.post('/embed/text', json={'queries': ['a dog']}).status_code == 401
    assert client.post('/embed/text', json={'queries': ['a dog']}, headers={'X-API-Key': 'wrong'}).status_code == 401
    assert client.post('/embed/text', json={'queries': ['  ']}, headers=headers).status_code == 422
    assert client.post('/embed/text', json={'queries': []}, headers=headers).status_code == 422
    too_many = [('files', ('a.jpg', b'x', 'image/jpeg'))] * 33
    assert client.post('/embed/images', files=too_many, headers=headers).status_code == 413
    too_big = [('files', ('a.jpg', b'x' * (8 * 1024 * 1024 + 1), 'image/jpeg'))]
    assert client.post('/embed/images', files=too_big, headers=headers).status_code == 413
    print('API contract: ok')

    found = find_coco()
    assert found, 'COCO is not attached, so accuracy was not measured.'
    with open(found[0]) as f:
        data = json.load(f)
    captions = {}
    for annotation in data['annotations']:
        captions.setdefault(annotation['image_id'], []).append(annotation['caption'].strip())
    images = sorted(data['images'], key=lambda image: image['id'])
    random.Random(0).shuffle(images)
    images = images[:SELF_TEST_IMAGES]
    jpegs = [thumbnail(os.path.join(found[1], image['file_name'])) for image in images]
    print(f'Thumbnail size: {np.mean([len(j) for j in jpegs]) / 1024:.0f} KB on average')

    # One unreadable file must not lose the rest of its batch.
    mixed = [('files', ('a.jpg', jpegs[0], 'image/jpeg')), ('files', ('bad.jpg', b'not an image', 'image/jpeg'))]
    vectors = client.post('/embed/images', files=mixed, headers=headers).json()['vectors']
    assert vectors[0] is not None and vectors[1] is None, vectors
    print('Bad image handling: ok')

    started = time.time()
    compressed, exact = [], []
    for start in range(0, len(jpegs), 16):
        batch = jpegs[start:start + 16]
        files = [('files', (f'{i}.jpg', data, 'image/jpeg')) for i, data in enumerate(batch)]
        response = client.post('/embed/images', files=files, headers=headers)
        assert response.status_code == 200, response.text
        compressed += [decode(v) for v in response.json()['vectors']]
    rate = len(jpegs) / (time.time() - started)
    print(f'Image embedding: {rate:.1f} images/s')
    for start in range(0, len(jpegs), 32):
        exact += embedder.embed_image_floats(jpegs[start:start + 32])

    texts, owners = [], []
    for index, image in enumerate(images):
        texts += captions[image['id']][:5]
        owners += [index] * len(captions[image['id']][:5])
    query_compressed, query_exact = [], []
    for start in range(0, len(texts), 16):
        batch = texts[start:start + 16]
        response = client.post('/embed/text', json={'queries': batch}, headers=headers)
        assert response.status_code == 200, response.text
        query_compressed += [decode(v) for v in response.json()['vectors']]
        query_exact += embedder.embed_texts([t.strip().lower() for t in batch])

    timings = []
    for i in range(20):
        started = time.time()
        client.post('/embed/text', json={'queries': [f'uncached query number {i}']}, headers=headers)
        timings.append(time.time() - started)
    print(f'Text search latency: {np.median(timings) * 1000:.0f} ms median')

    def recall(image_vectors, query_vectors):
        scores = torch.from_numpy(np.stack(query_vectors).astype(np.float32)) @ torch.from_numpy(np.stack(image_vectors).astype(np.float32)).T
        truth = torch.tensor(owners)
        ranks = (scores > scores.gather(1, truth[:, None])).sum(1).float()
        return {f'R@{k}': (ranks < k).float().mean().item() for k in (1, 5, 10)}

    exact_recall = recall(exact, query_exact)
    compressed_recall = recall(compressed, query_compressed)
    clipped = np.mean(np.abs(np.stack(compressed)) == 127)
    print('Uncompressed vectors:', exact_recall)
    print('Compressed vectors:  ', compressed_recall, f'({clipped:.4%} of values clipped)')
    assert compressed_recall['R@5'] >= exact_recall['R@5'] - 0.01, 'Compression hurts ranking too much'

    with open('self_test.json', 'w') as f:
        json.dump({
            'images': len(images), 'captions': len(texts), 'images_per_second': rate,
            'text_ms_median': float(np.median(timings) * 1000),
            'thumbnail_kb': float(np.mean([len(j) for j in jpegs]) / 1024),
            'uncompressed': exact_recall, 'compressed': compressed_recall, 'clipped': float(clipped),
        }, f, indent=2)
    print('SELF-TEST PASSED')

if RUN_SELF_TEST:
    self_test()

## Serve

Uses `NGROK_DOMAIN` for a fixed address if you have one, so the app never needs its URL changed.

In [ ]:
import uvicorn
from kaggle_secrets import UserSecretsClient
from pyngrok import conf, ngrok

def secret(name):
    try:
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

ngrok_token, api_key, domain = secret('NGROK_AUTH_TOKEN'), secret('SMART_GALLERY_API_KEY'), secret('NGROK_DOMAIN')

if not (ngrok_token and api_key):
    print('Secrets are missing, so the server was not started. See the first cell.')
else:
    app = create_app(embedder, api_key)
    server = uvicorn.Server(uvicorn.Config(app, host='127.0.0.1', port=8000, log_level='warning'))
    threading.Thread(target=server.run, daemon=True).start()
    conf.get_default().auth_token = ngrok_token
    tunnel = ngrok.connect(8000, 'http', **({'domain': domain} if domain else {}))
    print('SERVER URL:', tunnel.public_url)

    started = time.time()
    app.state.last_request = started
    while True:
        time.sleep(60)
        idle = time.time() - app.state.last_request
        hours = (time.time() - started) / 3600
        print(f'{hours:.1f}h up, {app.state.embedded_images} images embedded, idle {idle / 60:.0f} min', flush=True)
        if idle > IDLE_STOP_MINUTES * 60 or hours > MAX_HOURS:
            print('Stopping to save GPU quota.')
            break
    ngrok.kill()
    server.should_exit = True